# Predictive Analysis- Data Preprocessing
This notebook preprocesses our dataset and merges the 5 individual datasets into a master dataset, with target variables for 24-hour failure prediction (classification).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta

### Loading all datasets

In [3]:
DATA_PATH = "drive/Othercomputers/My Mac/predictive-analysis/data/raw/"

telemetry_df = pd.read_csv(f'{DATA_PATH}PdM_telemetry.csv', parse_dates=['datetime'])
errors_df = pd.read_csv(f'{DATA_PATH}PdM_errors.csv', parse_dates=['datetime'])
maintenance_df = pd.read_csv(f'{DATA_PATH}PdM_maint.csv', parse_dates=['datetime'])
failures_df = pd.read_csv(f'{DATA_PATH}PdM_failures.csv', parse_dates=['datetime'])
machines_df = pd.read_csv(f'{DATA_PATH}PdM_machines.csv')

### Data filtering
Our dataset has maintainance data from 2014 but other data (like failure, telementry) starts from 2015. So, we need to filter our maintainace data to match other dataset and start from 2015.

In [4]:
print(f"Before: {len(maintenance_df):,} rows")
print(f"Date range: {maintenance_df['datetime'].min()} to {maintenance_df['datetime'].max()}")

maintenance_df = maintenance_df[maintenance_df['datetime'] >= '2015-01-01']

print(f"After: {len(maintenance_df):,} rows")
print(f"Date range: {maintenance_df['datetime'].min()} to {maintenance_df['datetime'].max()}")

Before: 3,286 rows
Date range: 2014-06-01 06:00:00 to 2016-01-01 06:00:00
After: 2,886 rows
Date range: 2015-01-01 06:00:00 to 2016-01-01 06:00:00


### Preparing events data

In this part, we are processing events data (errors, maintenance, failures) for each machine at every hour, and creating 3 pivot tables where each row represents the event status of a specific machine at a specific time. These tables include binary flags for each event type showing whether any event occurred during that time.
<br>This structure makes it easier to analyze a machine's condition over time: "when & what event happened to this machine?" and will be merged into master dataset later.


In [5]:
# Processing ERROR events
# binary flags for each error type
errors_pivot = errors_df.pivot_table(
    index=['machineID', 'datetime'],
    columns='errorID',
    aggfunc='size',
    fill_value=0
).reset_index()

# renaming columns
errors_pivot.columns = ['machineID', 'datetime'] + [f'{col}' for col in errors_pivot.columns[2:]] # setting error type columns from pivoted table

# error flag
errors_pivot['has_error'] = (errors_pivot.iloc[:, 2:].sum(axis=1) > 0).astype(int)

print(f"Error pivot columns: {list(errors_pivot.columns)}")


Error pivot columns: ['machineID', 'datetime', 'error1', 'error2', 'error3', 'error4', 'error5', 'has_error']


In [6]:
# Processing MAINTENANCE events
maintenance_pivot = maintenance_df.pivot_table(
    index=['machineID', 'datetime'],
    columns='comp',
    aggfunc='size',
    fill_value=0
).reset_index()

# rename columns
maintenance_pivot.columns = ['machineID', 'datetime'] + [f'maint_{col}' for col in maintenance_pivot.columns[2:]]

# maintenance flag
maintenance_pivot['has_maintenance'] = (maintenance_pivot.iloc[:, 2:].sum(axis=1) > 0).astype(int)

print(f"Maintenance columns: {list(maintenance_pivot.columns)}")


Maintenance columns: ['machineID', 'datetime', 'maint_comp1', 'maint_comp2', 'maint_comp3', 'maint_comp4', 'has_maintenance']


In [7]:
# Processing FAILURE events
failures_pivot = failures_df.pivot_table(
    index=['machineID', 'datetime'],
    columns='failure',
    aggfunc='size',
    fill_value=0
).reset_index()

# rename columns
failures_pivot.columns = ['machineID', 'datetime'] + [f'failure_{col}' for col in failures_pivot.columns[2:]]

# failure flag
failures_pivot['has_failure'] = (failures_pivot.iloc[:, 2:].sum(axis=1) > 0).astype(int)

print(f"Failure columns: {list(failures_pivot.columns)}")

Failure columns: ['machineID', 'datetime', 'failure_comp1', 'failure_comp2', 'failure_comp3', 'failure_comp4', 'has_failure']


### Merging dataset

Here, we are merging all our datasets into one master dataset. <br>
We take the telementry dataset as the base table by making a copy of it first, then do a left join on each of our pivoted events table one by one on the columns of machineID and datetime. Any row with a missing event data will be assinged a 'NaN' value on merge so we replace the NaN values with 0, representing the event as false for that entry.

In [9]:
# making a copy of telemetry data as the base
master_df = telemetry_df.copy()

# merging with machines dataset (no datetime on this)
master_df = master_df.merge(machines_df, on='machineID', how='left')

# merging with errors pivot dataset, replacing all NaN values with 0
master_df = master_df.merge(errors_pivot, on=['machineID', 'datetime'], how='left')
error_cols = [col for col in master_df.columns if col.startswith('error') or col == 'has_error']
master_df[error_cols] = master_df[error_cols].fillna(0).astype(int)

# merging with maintenance pivot dataset, replacing all NaN values with 0
master_df = master_df.merge(maintenance_pivot, on=['machineID', 'datetime'], how='left')
maint_cols = [col for col in master_df.columns if col.startswith('maint_') or col == 'has_maintenance']
master_df[maint_cols] = master_df[maint_cols].fillna(0).astype(int)

# mergeing with failures pivot dataset, replacing all NaN values with 0
master_df = master_df.merge(failures_pivot, on=['machineID', 'datetime'], how='left')
failure_cols = [col for col in master_df.columns if col.startswith('failure_') or col == 'has_failure']
master_df[failure_cols] = master_df[failure_cols].fillna(0).astype(int)

master_df = master_df.sort_values(['machineID', 'datetime']).reset_index(drop=True)

In [11]:
print("\nColumn summary of merged dataset:")
print(master_df.dtypes)


Column summary of merged dataset:
datetime           datetime64[ns]
machineID                   int64
volt                      float64
rotate                    float64
pressure                  float64
vibration                 float64
model                      object
age                         int64
error1                      int64
error2                      int64
error3                      int64
error4                      int64
error5                      int64
has_error                   int64
maint_comp1                 int64
maint_comp2                 int64
maint_comp3                 int64
maint_comp4                 int64
has_maintenance             int64
failure_comp1               int64
failure_comp2               int64
failure_comp3               int64
failure_comp4               int64
has_failure                 int64
dtype: object
